In [1]:
# Install Dash and Plotly if not already present
# Run once in Terminal: pip install dash plotly

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, html, dcc, Input, Output

In [2]:
# Load the dashboard-ready CSV produced in 08_dashboard_export.ipynb
# This file contains original candidate data + ML predictions:
#   - placement_probability: XGBoost confidence score (0–1)
#   - placement_risk_tier: business label (High Confidence / Likely / At Risk / Unlikely)
#   - was_placed: actual historical outcome (1 = placed, 0 = not placed)

df = pd.read_csv('Recruitment_dashboard_ready.csv')

# Pre-compute headline KPIs used in the dashboard cards
total          = len(df)                                          # all registered candidates
placed         = int(df['was_placed'].sum())                      # historically placed
placement_rate = round(placed / total * 100, 1)                   # overall placement %
at_risk        = len(df[df['placement_risk_tier'].isin(          # model flagged as needing action
                    ['At Risk', 'Unlikely'])])

print(f"Total: {total} | Placed: {placed} | Rate: {placement_rate}% | At Risk: {at_risk}")

Total: 500 | Placed: 370 | Rate: 74.0% | At Risk: 73


In [10]:
# Initialise the Dash app — Dash is a Python framework that turns
# Plotly charts into a full interactive web dashboard without needing HTML/JS

app = Dash(__name__)

app.layout = html.Div(
    style={'backgroundColor': '#1B2A4A', 'padding': '20px', 'fontFamily': 'Arial'},
    children=[

    # ── HEADER ────────────────────────────────────────────────────────────────
    # Project title and model attribution line
    html.H1('Recruitment Firm — Recruitment Analytics Dashboard',
            style={'color': 'white', 'textAlign': 'center', 'marginBottom': '5px'}),
    html.P('Powered by XGBoost Placement Prediction | May 2026',
           style={'color': '#90CAF9', 'textAlign': 'center', 'marginBottom': '30px'}),

    # ── KPI ROW ───────────────────────────────────────────────────────────────
    # Four summary stat cards: Total | Placed | Rate | At Risk
    # Each card is a Div styled as a rounded dark box with a coloured metric number
    html.Div(
        style={'display': 'flex', 'justifyContent': 'space-around', 'marginBottom': '30px'},
        children=[
            html.Div(style={'backgroundColor': '#263B5E', 'borderRadius': '10px',
                            'padding': '20px', 'textAlign': 'center', 'width': '20%'}, children=[
                html.H2(str(total),  style={'color': '#90CAF9', 'margin': '0'}),
                html.P('Total Registered', style={'color': 'white', 'margin': '5px 0'})
            ]),
            html.Div(style={'backgroundColor': '#263B5E', 'borderRadius': '10px',
                            'padding': '20px', 'textAlign': 'center', 'width': '20%'}, children=[
                html.H2(str(placed), style={'color': '#4CAF50', 'margin': '0'}),
                html.P('Total Placed', style={'color': 'white', 'margin': '5px 0'})
            ]),
            html.Div(style={'backgroundColor': '#263B5E', 'borderRadius': '10px',
                            'padding': '20px', 'textAlign': 'center', 'width': '20%'}, children=[
                html.H2(f'{placement_rate}%', style={'color': '#FFD700', 'margin': '0'}),
                html.P('Placement Rate',  style={'color': 'white', 'margin': '5px 0'})
            ]),
            html.Div(style={'backgroundColor': '#263B5E', 'borderRadius': '10px',
                            'padding': '20px', 'textAlign': 'center', 'width': '20%'}, children=[
                html.H2(str(at_risk),    style={'color': '#F44336', 'margin': '0'}),
                html.P('At Risk / Unlikely', style={'color': 'white', 'margin': '5px 0'})
            ]),
        ]
    ),

    # ── FILTER SLICERS ────────────────────────────────────────────────────────
    # Two dropdown filters that feed the callback below.
    # Changing either dropdown triggers update_charts() to re-draw all 4 charts
    # and re-filter the at-risk table automatically.
    html.Div(style={'display': 'flex', 'gap': '20px', 'marginBottom': '20px'}, children=[
        html.Div([
            html.Label('Seniority Level', style={'color': 'white'}),
            dcc.Dropdown(
                id='seniority-filter',
                options=[{'label': 'All', 'value': 'All'}] +
                        [{'label': s, 'value': s}
                         for s in sorted(df['seniority_level'].dropna().unique())],
                value='All', clearable=False,
                style={'width': '200px'}
            )
        ]),
        html.Div([
            html.Label('Risk Tier', style={'color': 'white'}),
            dcc.Dropdown(
                id='tier-filter',
                options=[{'label': 'All', 'value': 'All'}] +
                        [{'label': t, 'value': t}
                         for t in ['High Confidence', 'Likely', 'At Risk', 'Unlikely']],
                value='All', clearable=False,
                style={'width': '200px'}
            )
        ]),
    ]),

    # ── CHARTS ROW 1 ──────────────────────────────────────────────────────────
    # Left: Donut chart showing pipeline risk tier breakdown
    # Right: Horizontal bar chart — placement rate by seniority level
    html.Div(style={'display': 'flex', 'gap': '20px', 'marginBottom': '20px'}, children=[
        html.Div(style={'width': '50%'}, children=[dcc.Graph(id='donut-chart')]),
        html.Div(style={'width': '50%'}, children=[dcc.Graph(id='seniority-bar')]),
    ]),

    # ── CHARTS ROW 2 ──────────────────────────────────────────────────────────
    # Left: Histogram of client satisfaction scores (placed candidates only)
    # Right: Scatter — salary gap vs satisfaction, coloured by seniority
    html.Div(style={'display': 'flex', 'gap': '20px', 'marginBottom': '20px'}, children=[
        html.Div(style={'width': '50%'}, children=[dcc.Graph(id='satisfaction-hist')]),
        html.Div(style={'width': '50%'}, children=[dcc.Graph(id='salary-scatter')]),
    ]),

    # ── AT-RISK TABLE ─────────────────────────────────────────────────────────
    # Candidate-level action table showing who needs recruiter follow-up
    # Sorted by placement_probability ascending (lowest confidence first)
    html.H3('At Risk & Unlikely Candidates — Action Required',
            style={'color': '#F44336', 'marginTop': '10px'}),
    html.Div(id='at-risk-table')
])

In [11]:
# ── CALLBACK ──────────────────────────────────────────────────────────────────
# The @app.callback decorator wires dropdowns → chart outputs.
# Every time a dropdown value changes, this function re-runs automatically.
# Inputs:  seniority-filter value, tier-filter value
# Outputs: all 4 chart figures + the at-risk table HTML

@app.callback(
    Output('donut-chart',       'figure'),
    Output('seniority-bar',     'figure'),
    Output('satisfaction-hist', 'figure'),
    Output('salary-scatter',    'figure'),
    Output('at-risk-table',     'children'),
    Input('seniority-filter',   'value'),
    Input('tier-filter',        'value')
)
def update_charts(seniority, tier):

    # Apply slicer filters — start from full dataset each call
    filtered = df.copy()
    if seniority != 'All':
        filtered = filtered[filtered['seniority_level'] == seniority]
    if tier != 'All':
        filtered = filtered[filtered['placement_risk_tier'] == tier]

    # ── CHART 1: Donut — Risk Tier Distribution ────────────────────────────
    # Shows how the current filtered pipeline splits across the 4 risk tiers
    # hole=0.5 creates the donut shape; color_map gives consistent tier colours
    tier_counts = filtered['placement_risk_tier'].value_counts().reset_index()
    tier_counts.columns = ['Tier', 'Count']
    color_map = {'High Confidence': '#4CAF50', 'Likely': '#2196F3',
                 'At Risk': '#FF9800', 'Unlikely': '#F44336'}
    donut = px.pie(tier_counts, names='Tier', values='Count', hole=0.5,
                   title='Pipeline Risk Distribution',
                   color='Tier', color_discrete_map=color_map)
    donut.update_layout(paper_bgcolor='#263B5E', font_color='white', title_font_size=14)

    # ── CHART 2: Horizontal Bar — Placement Rate by Seniority ─────────────
    # Grouped mean of was_placed (0/1) per seniority gives a % placement rate
    # Orientation='h' makes it horizontal — easier to read category labels
    sen = filtered.groupby('seniority_level')['was_placed'].mean().reset_index()
    sen.columns = ['Seniority', 'Placement Rate']
    sen['Placement Rate'] = (sen['Placement Rate'] * 100).round(1)
    bar = px.bar(sen, x='Placement Rate', y='Seniority', orientation='h',
                 title='Placement Rate by Seniority',
                 color='Placement Rate', color_continuous_scale='Blues')
    bar.update_layout(paper_bgcolor='#263B5E', plot_bgcolor='#263B5E',
                      font_color='white', title_font_size=14)

    # ── CHART 3: Histogram — Client Satisfaction (Placed Only) ────────────
    # Only placed candidates have a satisfaction score — filtering to was_placed=1
    # avoids 0-filled values from unplaced candidates distorting the distribution
    placed_only = filtered[filtered['was_placed'] == 1]
    hist = px.histogram(placed_only, x='client_satisfaction_score', nbins=10,
                        title='Client Satisfaction Distribution (Placed Only)',
                        color_discrete_sequence=['#2196F3'])
    hist.update_layout(paper_bgcolor='#263B5E', plot_bgcolor='#263B5E',
                       font_color='white', title_font_size=14)

    # ── CHART 4: Scatter — Salary Gap vs Satisfaction ─────────────────────
    # salary_gap = offered_salary - expected_salary (positive = paid above expectation)
    # A positive correlation here would mean exceeding salary expectations improves satisfaction
    # colour by seniority to spot whether this relationship differs by career level
    scatter = px.scatter(placed_only, x='salary_gap', y='client_satisfaction_score',
                         color='seniority_level',
                         title='Salary Gap vs Client Satisfaction',
                         labels={'salary_gap': 'Salary Gap (£)',
                                 'client_satisfaction_score': 'Satisfaction Score'})
    scatter.update_layout(paper_bgcolor='#263B5E', plot_bgcolor='#263B5E',
                          font_color='white', title_font_size=14)

    # ── AT-RISK TABLE ─────────────────────────────────────────────────────
    # Shows only At Risk + Unlikely candidates, sorted lowest probability first
    # so recruiters see the most urgent cases at the top
    at_risk_df = (
        filtered[filtered['placement_risk_tier'].isin(['At Risk', 'Unlikely'])]
        [['first_name', 'last_name', 'seniority_level',
          'expected_salary_gbp', 'placement_probability', 'placement_risk_tier']]
        .sort_values('placement_probability')
    )

    # Build table as Dash html components (no external table library needed)
    table = html.Table(
        style={'width': '100%', 'borderCollapse': 'collapse', 'color': 'white'},
        children=[
            html.Thead(html.Tr([
                html.Th(col, style={'padding': '10px', 'backgroundColor': '#263B5E',
                                    'borderBottom': '2px solid #90CAF9'})
                for col in ['First Name', 'Last Name', 'Seniority',
                            'Expected Salary', 'Placement Prob', 'Risk Tier']
            ])),
            html.Tbody([
                html.Tr([
                    html.Td(str(row[col]),
                            style={'padding': '8px', 'borderBottom': '1px solid #263B5E',
                                   'backgroundColor': '#1B2A4A'})
                    for col in at_risk_df.columns
                ]) for _, row in at_risk_df.iterrows()
            ])
        ]
    )

    return donut, bar, hist, scatter, table

In [12]:
# Launches the dashboard on http://localhost:8050
# debug=True enables hot-reload — any code change auto-refreshes the browser
# port=8050 is the default Dash port (avoid 8888/8889 used by JupyterLab)
#
# To view: open a new browser tab and go to http://localhost:8050
# To stop:  Kernel → Interrupt in JupyterLab, or press the ■ stop button

if __name__ == '__main__':
    app.run(debug=True, port=8050)
#    app.run(jupyter_mode='inline', port=8050)